# Automated Failure Mode Classification in Rock Mechanics

When a rock sample is crushed in a laboratory test, the way it breaks is recorded as a failure mode code - for example, 2B means a shear fracture at a specific angle, and XA means the sample broke along an existing crack. These codes are currently assigned by a person looking at the broken sample, which takes time and can vary between observers.

This project trains a machine learning model to predict the failure mode code from measurements already recorded during the test, such as the size of the sample, how much pressure was applied, and how stiff the rock was. If the model can do this reliably, it could speed up analysis and make the results more consistent.

The data comes from two types of laboratory tests: uniaxial compression (the sample is squeezed from top and bottom with no side pressure, about 177 samples) and triaxial compression (side pressure is added to simulate conditions deep underground, about 351 samples). A third test type in the raw data, the Brazilian tensile test, does not have failure mode codes and is not used for classification. A fourth dataset covering rock mass field observations is also excluded because it describes the rock at a tunnel or mine face, not individual laboratory samples.

Three classification methods are tested: Logistic Regression, Random Forest, and Support Vector Machine. The best one is tuned further, and the results are explained using a tool called SHAP, which shows which measurements had the most influence on each prediction.

## Section 1 - Libraries

This cell loads all the tools (called libraries) that are used throughout the notebook. Loading everything in one place at the start means the notebook will raise an error immediately if something is missing, rather than failing halfway through a later step. The version numbers are printed so that anyone trying to reproduce the results knows exactly which software was used.

In [2]:
# Install any missing libraries automatically before importing them.
# This means the notebook can run on any computer without manual setup steps.
import subprocess, sys

required = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "shap",
    "xlrd",
]

for package in required:
    try:
        __import__(package if package != "scikit-learn" else "sklearn")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"{package} installed.")

print("All libraries ready.")

# ----- Imports -----

import warnings
warnings.filterwarnings("ignore")

# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Interpretability
import shap

# Reproducibility - fixing the random seed means results are the same every time the notebook is run
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

# Print version numbers so results can be reproduced on another machine
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"matplotlib   {matplotlib.__version__}")
print(f"seaborn      {sns.__version__}")
import sklearn; print(f"scikit-learn {sklearn.__version__}")
print(f"shap         {shap.__version__}")

Installing shap...
shap installed.
Installing xlrd...
xlrd installed.
All libraries ready.
numpy        1.26.4
pandas       2.2.3
matplotlib   3.9.2
seaborn      0.13.2
scikit-learn 1.5.1
shap         0.51.0
